# Notebook 4: Deploy Online Inference Service + Test

This notebook deploys `WELL_HEALTH_MODEL` from the Model Registry as an online inference service on Snowpark Container Services (SPCS), then validates it with both Python SDK and SQL calls.

**Pipeline Position:** Final step — takes the trained model (Notebook 3) and makes it available as a real-time REST endpoint for health score predictions.

**Architecture:**
- Model served via SPCS as a containerized REST endpoint
- Accessible through Python SDK (`mv.run`) and direct SQL function calls
- Functions exposed: `PREDICT`, `DECISION_FUNCTION`, `SCORE_SAMPLES`

## 1. Connect and Configure

Establish a Snowpark session and load configuration constants (model name, version, compute pool, inference service name) from shared utilities.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DATABASE = "ENERGY_DEMO"
SCHEMA = "WELLS"
COMPUTE_POOL = "COCO_ML_COMPUTE_POOL"
MODEL_NAME = "WELL_HEALTH_MODEL"
SERVICE_NAME = "WELL_HEALTH_INFERENCE_SERVICE"

session.use_database(DATABASE)
session.use_schema(SCHEMA)
print(f"Connected to {DATABASE}.{SCHEMA}")

## 2. Verify Model in Registry

Confirm the trained model exists in the Snowflake Model Registry before attempting to deploy it. This retrieves the model version object that we'll use to create the service.

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session)
model = registry.get_model(MODEL_NAME)
versions = model.versions()
print(f"Model: {MODEL_NAME}")
print(f"Available versions: {[v.version_name for v in versions]}")
mv = versions[-1]
MODEL_VERSION = mv.version_name
print(f"Using latest version: {MODEL_VERSION}")
fns = mv.show_functions()
print(f"Functions: {[f['name'] for f in fns]}")

## 3. Create Inference Service

Deploy the model as a containerized REST endpoint on SPCS. This builds a container image from the registered model and starts it on the compute pool. Initial deployment takes ~5-10 minutes.

In [ ]:
try:
    session.sql(f"DROP SERVICE IF EXISTS {SERVICE_NAME}").collect()
except Exception:
    pass

print(f"Creating {SERVICE_NAME} on {COMPUTE_POOL}...")
mv.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool=COMPUTE_POOL,
    ingress_enabled=True,
    max_instances=1,
    autocapture=True,
    build_external_access_integrations=["PYPI_ACCESS_INTEGRATION"],
)
print("Service creation initiated!")

## 4. Test Inference via Python SDK (`mv.run`)

Run a batch prediction using the Python SDK. This passes a DataFrame of feature values to the service and returns anomaly scores — the primary interface for notebook-based workflows.

In [ ]:
df_test = session.sql("""
    SELECT INTAKE_PRESSURE_PSI, DISCHARGE_PRESSURE_PSI, PRESSURE_DIFFERENTIAL,
           MOTOR_TEMP_F, MOTOR_AMPS, VIBRATION_IPS,
           WELLHEAD_PRESSURE_PSI, WELLHEAD_TEMP_F, FREQUENCY_HZ,
           AVG_OIL_7D AS AVG_OIL, AVG_GAS_7D AS AVG_GAS,
           AVG_WATER_7D AS AVG_WATER, AVG_RUNTIME_7D AS AVG_RUNTIME,
           WATER_CUT_PCT AS AVG_WATER_CUT, GOR AS AVG_GOR
    FROM WELL_HEALTH_FEATURES LIMIT 5
""").to_pandas()

result = mv.run(df_test, function_name="predict", service_name=SERVICE_NAME)
print(f"Predictions: {result.shape}")
print("HEALTH_SCORE: 0.0 = highly anomalous, 1.0 = healthy")
result